# Grün ist nicht gleich Grün - Vegetationsanalyse mit Sentinel-2

Bisher habt ihr ein paar grundlegende Programmcodes kennengelernt. Als nächstes wird das ganze mal an einem Beispiel betrachtet. In diesem Teil lernt ihr unter anderem, wie man Satellitenaufnahmen von Sentinel-2 in einer Karte visualisiert. Dazu werdet ihr den Unterschied zwischen Echt- und Falschfarbenbildern kennenlernen. Als Beispiel betrachten wir Vegetation in Berlin, also alles rund um Pflanzen.

# 1 Pakete importieren

In [ ]:
import matplotlib.pyplot as plt    # Für Diagramme und Visualisierungen
import numpy as np                 # Für numerische Operationen mit Arrays (Datacubes)
import os                          # Für Betriebssystem-Funktionen (z.B. Dateipfade)
import rasterio                    # Zum Lesen/Schreiben von GeoTIFF-Dateien
import leafmap                     # Interaktive Karte erstellen

from geolibre import Map           # Für Geodaten-Visualisierung auf Karten

# 2 Vegetation in Berlin untersuchen

Für die Untersuchung von Klima und Klimawandel kann die Vegetation herangezogen werden, da sie unmittelbar auf Temperaturunterschiede und Trockenheit reagiert. Diese Reaktionen lassen sich auch in Satellitenbildern messen. Wenn Vegetation beobachtet werden soll, muss zuerst festgestellt werden, wo sie sich befindet. Dies scheint offensichtlich, doch nicht alles, was wie Vegetation aussieht, ist tatsächlich Vegetation. Dies wird am Beispiel einiger Sportplätze in Berlin genauer betrachtet.

:::{.task}

#### Aufgabe 1

Aktiviert die interaktive Karte und schaut euch in dem Kartenausschnitt etwas um. Beschreibt, was ihr auf der Karte sehen könnt. Da es sich um hochaufgelöste Aufnahmen handelt, könnt ihr sehr nah ranzoomen und somit auch Details näher betrachten.

*(5 Minuten)*
:::

In [ ]:
# Neue Karte konfigurieren mit Koordinaten und Zoomstufe; die Karte wird als Variable my_map gespeichert
my_map = leafmap.Map(center=[52.545154, 13.399177], zoom=15)

# Basemap hinzufügen
my_map.add_basemap("Esri.WorldImagery")

# Zuvor konfigurierte Karte aufrufen
my_map

Es sind Sportplätze sichtbar, die kräftig grün erscheinen, genauso wie die vielen Bäume und Pflanzen drumherum. Beides würde ohne zu zögern als Vegetation bezeichnet werden. Doch einige der Plätze sind Kunstrasenplätze und somit keine echte Vegetation. Für eine sichere Unterscheidung zwischen echter Vegetation und Kunstrasen werden jedoch mehr Daten benötigt.

## 2.1 Echtfarbenbild

Damit ihr zwischen echter Vegetation und Kunstrasen unterscheiden könnt, braucht es zuerst ein Echtfarbenbild. Hierfür werden Sentinel-2 Daten genutzt. Beachtet, dass der Zeitraum, aus dem die Bilder stammen, sinnvoll gewählt wird (idealerweise wird die Vegetationsperiode von März bis September eines Jahres gewählt, in unserem Beispiel haben wir einen Zeitraum von Mai bis August gewählt).

In [ ]:
# Visualisierung RGB
with rasterio.open("./data/berlin_rgb_cog.tif") as src:
    rgb_data = src.read([1, 2, 3])                      # Sentinel-2 L2A Daten sind oft in DN (Digital Numbers) skaliert (0-10000)
    rgb_data = np.clip(rgb_data / 3000.0, 0, 1)         # Einfache Skalierung für die Anzeige (0-3000 als Bereich für bessere Kontraste)
    rgb_display = np.transpose(rgb_data, (1, 2, 0))     # Umordnen von (Bands, H, W) zu (H, W, Bands) für imshow

plt.figure(figsize=(10, 10))
plt.imshow(rgb_display)
plt.title("Echtfarben (True Color RGB) - Berlin")       # Titel der Grafik, kann von euch umbenannt werden
plt.axis("off")
plt.show()
# plt.savefig("rgb")

Auf den ersten Blick erscheinen die Daten schlechter als bei der interaktiven Karte. Dies liegt daran, dass ein Pixel nur noch einem Quadrat von 10m x 10m entspricht und dadurch die **räumliche Auflösung** sinkt. Dabei gehen Details verloren, doch Fußballplätze bleiben weiterhin sichtbar.

Die Farben bleiben grundsätzlich unverändert, da es sich immer noch um ein Echtfarbenbild handelt. Dies entsteht folgendermaßen: Der Bildschirm mischt alle Farben aus Rot, Grün und Blau (RGB). Deshalb wird ein Echtfarbenbild auch oftmals **RGB-Bild** genannt. Dem roten Farbkanal wird das rote reflektierte Licht zugeordnet, das von Sentinel-2 in Band 4 aufgezeichnet wurde. Dem grünen Kanal (Band 3) wird das grüne Licht und dem blauen Kanal (Band 2) das blaue Licht zugeordnet.
- R = Rotes Licht (B4)
- G = Grünes Licht (B3)
- B = Blaues Licht (B2)

Bei jedem Foto, das auf einem Smartphone angezeigt wird, passiert dasselbe. Filter verändern dann die Kanalkombination, um mehr Kontrast oder lustige Farbeffekte zu erzeugen.

## 2.2 Falschfarbenbild

Die RGB-Farben sind Farben des sichtbaren Spektrums, also Licht, das wir sehen können. Satelliten wie Sentinel-2 nehmen mehr Wellenlängenbereiche auf, als das menschliche Auge sehen kann, z. B. infrarotes Licht (bekannt von Fernbedienungen oder Wärmelampen), Mikrowellen (Radar) oder Ultraviolett. Dieses für uns unsichtbare Licht kann auf einen der drei Farbkanäle gelegt und so sichtbar gemacht werden.

In der nächsten Codezelle wird erneut ein Satellitenbild geladen, allerdings werden andere Bereiche des reflektierten Lichts ausgewählt. Band 8 des Sentinel-2 deckt Wellenlängen des **nahen Infrarots** ab, deshalb wird dieses Band für den roten Kanal gewählt. Der grüne Kanal deckt rotes Licht und der blaue Kanal grünes Licht ab. Blaues Licht wird nicht dargestellt. Dadurch entsteht ein sogenanntes **Falschfarbenbild**, auch **CIR-Bild** genannt (CIR steht für Color-Infrared).

- R = Infrarotes Licht (B8)
- G = Rotes Licht (B4)
- B = Grünes Licht (B3)

:::{.callout-note}
#### Hinweis

Wieso genau Infrarot und rotes Licht bei der Untersuchung von Vegetation gewählt werden, erfahrt ihr im dritten Teilmodul.

:::

In [ ]:
# Visualisierung CIR
with rasterio.open("./data/berlin_cir_cog.tif") as src:
    cir_data = src.read([1, 2, 3])                      # Bänder 1, 2, 3 entsprechen hier B08, B04, B03
    cir_data = np.clip(cir_data / 3000.0, 0, 1)
    cir_display = np.transpose(cir_data, (1, 2, 0))

plt.figure(figsize=(10, 10))
plt.imshow(cir_display)
plt.title("Falschfarbenbild (CIR) - Berlin")
plt.axis("off")
plt.show()
# plt.savefig("cir")

## 2.3 Abgleich mit einer Landbedeckungsklassifikation

Damit ihr eure Vermutungen bezüglich Kunstrasen überprüfen könnt, wird eine Landbedeckungsklassifikation der europäischen Raumfahrtbehörde ESA genutzt (ESA Worldcover 2020).

Hier seht ihr wie eine solche Klassifikationskarte für den Großraum Berlin aussieht. Die Naturrasenplätze werden gelb dargestellt, was der Klasse Grassland entspricht. Die Kunstrasenplätze werden der Klasse Built-up zugeordnet. Diese steht zwar eigentlich für Bebauung, umfasst jedoch nicht nur Gebäude, sondern generell versiegelte bzw. künstliche Materialien wie Asphalt, Beton, Dachziegel sowie künstliche Oberflächen wie Kunststoffbahnen auf Sportplätzen oder Kunstrasen.


![ESA World Cover Berlin](./images/CDEC_M1_Teil2_esa-wc-berlin.png)

[Bildquelle](https://esa-worldcover.org/en/data-access)

![ESA World Cover Legende](./images/CDEC_M1_Teil2_esa-wc-legend.png)

[Bildquelle](https://esa-worldcover.s3.eu-central-1.amazonaws.com/v100/2020/docs/WorldCover_PUM_V1.0.pdf)

| Farbcode / Darstellung | Name (Englisch) | Name (Deutsch) |
| --- | --- | --- |
| **Dunkelgrün** | Tree cover | Baumbedeckung |
| **Orange** | Shrubland | Strauchvegetation |
| **Hellgelb** | Grassland | Grasland |
| **Rosa / Violett** | Cropland | Ackerland |
| **Rot** | Built-up | Bebaute Fläche |
| **Grau** | Bare / sparse vegetation | Karg / Spärliche Vegetation |
| **Weiß** | Snow and ice | Schnee / Eis |
| **Blau** | Permanent water bodies | Wasserflächen |
| **Türkis** | Herbaceous wetland | Feuchtgebiet |
| **Hellgrün** | Mangroves | Mangroven |
| **Khaki** | Moss and lichen | Moose / Flechten |

# 3 Vergleich der Satellitenbilder

Mit dem nachfolgenden Code werden die Bilder in eine interaktiven Karte eingeladen und visualisiert. Wir nutzen diesmal nicht leafmap, sondern *GeoLibre*, weil wir ein paar Funktionen benötigen, die leafmap nicht hat. Wenn ihr den Code ausführt, öffnet sich eine interaktive Karte mit einer Layer-Swipe Funktion (im Code *Split-Map* genannt). Die Dateien werden automatisch in die Karte geladen. Ihr braucht nichts weiter machen und könnt jetzt mit den Bildern interagieren.

:::{.callout-note}
#### Hinweis

Weitere Einstellungen für die Layer-Swipe Funktion sind im Kartenfenster über das "Pause-Zeichen" oben links zu erreichen.

:::

:::{.task}

#### Aufgabe 2

Nutzt den Slider in der Kartenmitte, um zwischen Echt- und Falschfarbenbild hin- und herzublenden, und beobachtet, wie die Vegetation anfängt rot zu leuchten. Wie viele Kunstrasenplätze entdeckt ihr im angezeigten Bildausschnitt? Beschreibt, wie ihr die Kunstrasenplätze identifiziert habt und vergleicht eure Ergebnisse mit der Landbedeckungsklassifikation.

*(15 Minuten)*
:::

In [ ]:
# RGB, CIR und ESA Daten mit Geolibre in einer Karte visualisieren
m = Map(center=[13.400207, 52.542231], zoom=13)    # [Lon, Lat]

rgb = m.add_cog("./data/berlin_rgb_cog.tif", name="RGB")
cir = m.add_cog("./data/berlin_cir_cog.tif", name="CIR")
esa = m.add_cog("./data/berlin_esa_cog.tif", name="ESA")

# Aktivieren der Split-Map bzw. Layer-Swipe Funktion
m.split_map(rgb, cir)
m

Warum erscheinen Vegetation und Kunstrasenplätze im Echtfarbenbild beide grün? Das liegt daran, dass sich beide Oberflächen im sichtbaren Licht sehr ähnlich verhalten. Sie absorbieren blaues und rotes Licht stark und reflektieren grünes Licht. Bei Pflanzen ist Chlorophyll für die grüne Farbe verantwortlich, bei Kunstrasen dagegen sind es grüne Farbpigmente.

Aber warum unterscheiden sich die Farben dann im Falschfarbenbild? Das Chlorophyll in den Pflanzen reflektiert infrarotes Licht sehr stark, weshalb das Bild plötzlich so rot leuchtet. Versiegelte Flächen oder solche mit Erdboden, wie Straßen oder abgeerntete Felder, reflektieren dieses Licht wiederrum nicht und erscheinen dunkel. So ist das auch bei gestresster Vegetation. Durch fehlendes Chlorophyll reflektiert die Pflanze infrarotes Licht weniger stark. Dieses Verhalten wird genutzt, um gesunde von geschwächter Vegetation zu unterscheiden.

![Reflektanzkurve](images/CDEC_M1_Teil3_reflektanzkurven.jpg)

[Bildquelle](https://ai.hdm-stuttgart.de/news/2022/satellitendaten-verstehen-die-grundlagen/)


:::{.callout-note}
#### Zusammenfassung

- **Echte, gesunde Vegetation**: Gesunde Pflanzen reflektieren nahes Infrarot stark (bis zu 50–60 % der Strahlung), da sie Chlorophyll enthalten.
- **Gestresste Vegetation**: Ist die Pflanze geschwächt, sinkt der Chlorophyllgehalt und die Reflektion im Infrarotbereich wird weniger.
- **Kunstrasen**: Plastik besitzt keine pflanzlichen Zellen. Es reflektiert im nahen Infrarot kaum mehr als im sichtbaren Licht.
- **Asphalt / Beton**: Mineralische Oberflächen reflektieren sichtbares Licht gleichmäßig mittelstark, aber ebenfalls kein nahes Infrarot.
:::

| Objekt | RGB [B04, B03, B02] | CIR [B08, B04, B03] |
| --- | --- | --- |
| **Gesunde Vegetation** | grün | leuchtendes rot / magenta |
| **Gestresste Vegetation** | gelb bis braun | dunkelrot, braun |
| **Kunstrasen** | sattes grün | dunkelgrau, braun bis blau-schwarz |
| **Bäume & Sträucher** | dunkelgrün | kräftiges rot bis rosa |
| **Asphalt / Beton** | grau bis weiß | cyan, hellgrau bis bläulich |


# 4 Zusammenfassung

In diesem Notebook habt ihr gelernt:
- was es mit Echt- und Falschfarbenbildern auf sich hat
- mithilfe eines Falschfarbenbildes Kunstrasen von echter Vegetation zu unterscheiden
- eure Vermutungen mit einer Klassifikationskarte zu bestätigen
- mit einer Layer-Swipe Funktion die verschiedenen Bildern zu vergleichen
- wie sich die Strahlungseigenschaften von verschiedenen Oberflächen verhalten